# Plotting, and how a chart misleads you

MichAl Academy, lesson 1.6.

Run each cell with **Shift+Enter**. matplotlib is already installed in Colab and
Kaggle.

Two halves. The first is the mechanics, which are short. The second is the part
that matters: the same true numbers drawn to say two different things.

## 1. The mechanics

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

MONTHS  = list("JFMAMJJASOND")
BLOCKED = [430, 425, 412, 418, 423, 431, 421, 439, 448, 455, 462, 470]

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar(MONTHS, BLOCKED)
ax.set_ylabel("phishing blocked")
ax.set_title("Blocked per month")
plt.show()

`plt.subplots()` hands back a figure and one or more axes. Everything you draw
and everything you label goes on an `ax`.

Learn it in this form rather than the `plt.bar()` shorthand. The moment you want
two charts side by side, the shorthand stops working and this does not.

In [ ]:
rng = np.random.default_rng(7)

hosts = pd.DataFrame({
    "length":   rng.integers(8, 40, 300),
    "age_days": rng.integers(1, 2000, 300),
})
hosts["flagged"] = (hosts["age_days"] < 200) & (hosts["length"] > 20)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))

hosts["length"].hist(bins=20, ax=axes[0])
axes[0].set_title("histogram: shape of one column")

hosts["flagged"].value_counts().plot(kind="bar", ax=axes[1])
axes[1].set_title("bar: counts per category")

hosts.plot.scatter(x="length", y="age_days", s=6, alpha=0.5, ax=axes[2])
axes[2].set_title("scatter: two columns against each other")

plt.tight_layout()
plt.show()

Four types cover nearly everything: **histogram** for the shape of one column,
**bar** for counts per category, **line** for something over time, **scatter**
for two columns against each other.

Reach for a pie chart never.

## 2. Where does the y axis start?

The same twelve numbers, drawn twice.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))

axes[0].bar(MONTHS, BLOCKED)
axes[0].set_ylim(0, 500)
axes[0].set_title("y axis from 0")

axes[1].bar(MONTHS, BLOCKED, color="tab:red")
axes[1].set_ylim(400, 480)
axes[1].set_title("y axis from 400")

plt.tight_layout()
plt.show()

change = (BLOCKED[-1] - BLOCKED[0]) / BLOCKED[0] * 100
print(f"first month {BLOCKED[0]}, last month {BLOCKED[-1]}, change {change:+.1f}%")

One chart says nothing happened. The other says something is badly wrong. The
number underneath both is the same.

A bar's whole job is that its length means its value, and that only works from
zero. Cut the axis and you are drawing *differences* while still using a shape
that says *quantity*.

A line chart of a stable quantity is the honest exception, because a line means
"how it moved" rather than "how much". Even then, label it.

## 3. What is it out of?

A security report says phishing blocked is up 40% year on year.

In [ ]:
first = {"blocked":  21_000, "emails": 1_000_000}
last  = {"blocked":  29_400, "emails": 1_380_000}

pct = lambda a, b: (b - a) / a * 100

print(f"blocked   {first['blocked']:>9,} -> {last['blocked']:>9,}   {pct(first['blocked'], last['blocked']):+.1f}%")
print(f"emails    {first['emails']:>9,} -> {last['emails']:>9,}   {pct(first['emails'], last['emails']):+.1f}%")

rate_first = first["blocked"] / first["emails"] * 100
rate_last  = last["blocked"]  / last["emails"]  * 100
print()
print(f"share that was phishing   {rate_first:.2f}% -> {rate_last:.2f}%   {pct(rate_first, rate_last):+.1f}% relative")

Almost all of the rise was more email arriving. The share that was phishing
barely moved, and that is the number that says whether anything changed.

This is the base-rate habit from lesson 0.4 in chart form. A chart is where it
will actually catch you, because a chart looks like it has already done the
thinking for you.

## 4. An average is one number standing in for a shape

Two columns. Same mean, same standard deviation.

In [ ]:
rng = np.random.default_rng(11)

def rescale(x, mean, sd):
    return (x - x.mean()) / x.std() * sd + mean

single = rescale(rng.normal(size=400), 50, 12)
split  = rescale(np.concatenate([rng.normal(-1, 0.3, 200),
                                 rng.normal(1, 0.3, 200)]), 50, 12)

for name, col in [("single", single), ("split", split)]:
    print(f"{name:7s} mean {col.mean():6.2f}   sd {col.std():5.2f}   median {np.median(col):6.2f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.4), sharex=True, sharey=True)

axes[0].hist(single, bins=30)
axes[0].set_title("single: mean 50, sd 12")

axes[1].hist(split, bins=30, color="tab:red")
axes[1].set_title("split: mean 50, sd 12")

plt.tight_layout()
plt.show()

Identical summary statistics, and one of them has no values anywhere near 50.

If those were response times, the left one is a system that is usually fine and
the right one is two different systems averaged together. Draw the histogram
before you trust the mean. It costs one line.

## 5. Your turn

Here is a column of domain ages from a real-looking export. The summary looks
unremarkable.

Find what is wrong with it, then fix the mean.

In [ ]:
rng = np.random.default_rng(3)

export = pd.DataFrame({
    "host": [f"host{i:03d}.example" for i in range(200)],
    "age_days": np.concatenate([
        rng.integers(200, 2000, 170),
        np.full(30, -1),          # what the exporting system writes for "unknown"
    ]),
})
export = export.sample(frac=1, random_state=0).reset_index(drop=True)

print(export["age_days"].describe())

In [ ]:
# TODO: draw the histogram, then compute the mean of only the real values
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.hist(export["age_days"], bins=40)
ax.set_title("age_days")
plt.show()

reported = export["age_days"].mean()
honest   = export["age_days"].mean()   # TODO: change this line

print(f"reported mean {reported:8.1f}")
print(f"honest mean   {honest:8.1f}")
print("different?", round(reported, 1) != round(honest, 1))

Look at the far left of the histogram, then look at `min` in the summary above.

<details>
<summary>Answer</summary>

Thirty rows hold `-1`, which is not an age. It is what the exporting system
writes when it does not know, and no age can be negative. Because `-1` is a
number, `describe()` happily averaged it in and dragged the mean down.

```python
honest = export.loc[export["age_days"] >= 0, "age_days"].mean()
```

The general habit: after loading anything, plot the distribution of every column
you intend to model on. Placeholder values, clipped ranges and duplicated
imports all show up instantly in a histogram and are invisible in a mean.

Better still, turn the placeholder into a real missing value so nothing can
average it by accident, which puts you back on the NaN rules from lesson 1.4:

```python
export["age_days"] = export["age_days"].replace(-1, np.nan)
```

</details>

## What you now have

- `fig, ax = plt.subplots()`, then everything on the `ax`
- Histogram, bar, line, scatter. Never a pie chart
- Bars need a y axis from zero, because their length is the message
- A count means nothing until you know what it was out of
- Two very different shapes can have the same mean, so plot the shape
- Plot every column before modelling on it, because that is how you find the `-1`

Next is lesson 1.7, just enough linear algebra, where similarity turns out to be
an angle.